### 加载数据

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
#from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")

In [2]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 市场基准
bench_symbol = "510300.SH"
bench = prices[bench_symbol]
prices = prices.drop(columns=[bench_symbol])  # 基准移出资产池
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)
# 中性化
X_net = X.sub(prices_to_returns(bench.to_frame("bench"), drop_inceptions_nan=False)["bench"], axis=0)
# Inf值检查
inf_cols = X_net.columns[np.isinf(X_net).any(axis=0)]
print(inf_cols.tolist())
# 异常收益检查
X_net = X_net.drop(columns=(bad := X_net.columns[(X_net.abs() > 0.25).any()])); 
X = X.drop(columns=(bad := X.columns[(X.abs() > 0.25).any()])); 
print(f"数据异常：收益率超过±30%的列已剔除 {len(bad)} 列 -> {bad.tolist()}")

[]
数据异常：收益率超过±30%的列已剔除 3 列 -> ['161811.SZ', '510030.SH', '511580.SH']


### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

中性化后相关性分布整体下移（中位数从高正相关降至 0.143）且正负两端均出现变化：强正相关配对减少约 40%，负相关配对从 0 增至千级。原参数网格（nondomin −0.3~−0.5 空转、correlate 0.1~0.5 高剔除）是按"市场因子主导的强正相关世界"标定的，直接复用确实不适配。建议：nondomin__threshold 收敛至 [-0.5, -0.4]、correlate__threshold 下调至 [0.15, 0.3]，同时计算规模恢复到原始口径水平。

## 相关性分布实测

| 配对相关性分布 | 原始 X | 超额 X_net | 变化 |
|---------------|--------|-----------|------|
| corr > 0.5 | 12,668 对 | 7,791 对 | **−38.5%** |
| corr > 0.3 | 27,712 对 | 16,726 对 | **−39.6%** |
| corr > 0.1 | 42,009 对 | 43,438 对 | +3.4% |
| corr < −0.3 | 0 对 | 1,269 对 | 0 → 1,269 |
| 中位数相关性 | 高（正相关主导） | 0.143 | 整体下移 |

In [3]:
import optuna
from skfolio import Population,MultiPeriodPortfolio
from skfolio import RiskMeasure,PerfMeasure,RatioMeasure,ExtraRiskMeasure
from sklearn.pipeline import Pipeline
from skfolio.metrics import make_scorer
from skfolio.optimization import EqualWeighted
from Pre_selection import DropTailCorrelated
from skfolio.pre_selection import SelectKExtremes
from skfolio.pre_selection import DropZeroVariance, DropCorrelated
from skfolio.pre_selection import SelectComplete, SelectNonExpiring, SelectNonDominated
from skfolio.model_selection import WalkForward
from skfolio.model_selection import cross_val_predict
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

### WalkForward交叉验证
- 数据泄露防护与执行延迟控制
通过 purged_size 参数控制训练集和测试集之间的“清洗”间隔，以模拟真实的交易执行延迟。
    - purged_size=0：训练结束与测试开始无缝衔接。
    - purged_size >= 1：在训练集末尾和测试集开头之间丢弃指定数量的观测值。
    - 建议：对于每日定价资产、流动性较差的市场或收盘后结算的数据，建议使用 purged_size >= 1 以更真实地反映执行延迟。

- 训练集扩展与尾部数据处理
    - expand_train=True：后续的训练集将包含所有过去的观测值，而不仅仅是固定长度的窗口。
    - reduce_test=True：即使最后一个测试集的样本数少于 test_size，也会返回该分割。默认情况下，不完整的测试集会被忽略。
#### WalkForward+GridSearch/RandomSearch/Optuna
以最大化样本外的平均指标（如 Mean-CVaR 比率）寻找最优参数。

In [ ]:
fitness_measures = [
[PerfMeasure.MEAN, RiskMeasure.VARIANCE],
[PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE],
[PerfMeasure.MEAN, RiskMeasure.STANDARD_DEVIATION],
[PerfMeasure.MEAN, RiskMeasure.SEMI_DEVIATION],
[PerfMeasure.MEAN, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
[PerfMeasure.MEAN, RiskMeasure.CVAR],
[PerfMeasure.MEAN, RiskMeasure.EVAR],
[PerfMeasure.MEAN, RiskMeasure.WORST_REALIZATION],
[PerfMeasure.MEAN, RiskMeasure.CDAR],
[PerfMeasure.MEAN, RiskMeasure.MAX_DRAWDOWN],
[PerfMeasure.MEAN, RiskMeasure.AVERAGE_DRAWDOWN],
[PerfMeasure.MEAN, RiskMeasure.EDAR],
[PerfMeasure.MEAN, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
[PerfMeasure.MEAN, RiskMeasure.ULCER_INDEX],
[PerfMeasure.MEAN, RiskMeasure.GINI_MEAN_DIFFERENCE],]


In [8]:
fitness_measures = [
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO],
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO],
[PerfMeasure.MEAN, RatioMeasure.MEAN_ABSOLUTE_DEVIATION_RATIO],
[PerfMeasure.MEAN, RatioMeasure.FIRST_LOWER_PARTIAL_MOMENT_RATIO],
[PerfMeasure.MEAN, RatioMeasure.VALUE_AT_RISK_RATIO],
[PerfMeasure.MEAN, RatioMeasure.CVAR_RATIO],
[PerfMeasure.MEAN, RatioMeasure.ENTROPIC_RISK_MEASURE_RATIO],
[PerfMeasure.MEAN, RatioMeasure.EVAR_RATIO],
[PerfMeasure.MEAN, RatioMeasure.WORST_REALIZATION_RATIO],
[PerfMeasure.MEAN, RatioMeasure.DRAWDOWN_AT_RISK_RATIO],
[PerfMeasure.MEAN, RatioMeasure.CDAR_RATIO],
[PerfMeasure.MEAN, RatioMeasure.CALMAR_RATIO],
[PerfMeasure.MEAN, RatioMeasure.AVERAGE_DRAWDOWN_RATIO],
[PerfMeasure.MEAN, RatioMeasure.EDAR_RATIO],
[PerfMeasure.MEAN, RatioMeasure.ULCER_INDEX_RATIO],
[PerfMeasure.MEAN, RatioMeasure.GINI_MEAN_DIFFERENCE_RATIO],]

In [27]:
fitness_measures = [
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.VARIANCE],
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.CDAR],
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.CVAR],
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.EDAR],
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.EVAR],
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.ULCER_INDEX],
[PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_VARIANCE],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_VARIANCE, ExtraRiskMeasure.KURTOSIS],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN, ExtraRiskMeasure.KURTOSIS],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.CDAR],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.CVAR],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.EDAR],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.EVAR],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.ULCER_INDEX],
[PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
[PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.CVAR],
[PerfMeasure.MEAN, RiskMeasure.VARIANCE, ExtraRiskMeasure.KURTOSIS],
[PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.CVAR],
[PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],
[PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],]

#### WalkForward+GridSearch

In [68]:
def combined_score(pred):
    """组合评分：年化收益 × (1 + 偏度)，负偏度惩罚、正偏度奖励"""
    return pred.annualized_mean + pred.skew - pred.max_drawdown - pred.kurtosis/100

In [69]:
cv = WalkForward(test_size=252//4, train_size=int(252*2), purged_size=1, reduce_test=True, expand_train=False)
model = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated()),
        ("correlate", DropCorrelated()),
        ("optimization", EqualWeighted())
    ])

param_grid = {
    "nondomin__min_n_assets" : [5, 10, 15, 20],
    "nondomin__threshold" : [-0.5, -0.4],
    "nondomin__fitness_measures" : [[PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],],
    "correlate__threshold" : [0.1, 0.2, 0.3],
}
grid_search = GridSearchCV(
    estimator=model,
    cv=cv,
    param_grid=param_grid,
    scoring=make_scorer(combined_score),
    n_jobs=8,
    verbose=2,
    refit=False,

)
grid_search.fit(X)
print(grid_search.best_params_)
print(grid_search.best_score_)

Fitting 33 folds for each of 24 candidates, totalling 792 fits
{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Variance, Average Drawdown], 'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.4}
0.04872835163593954


#### WalkForward+RandomSearch

In [ ]:
cv = WalkForward(test_size=252, train_size=int(252*3), purged_size=1, reduce_test=True, expand_train=False)
model = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated()),
        ("correlate", DropCorrelated()),
        ("optimization", EqualWeighted())
    ])

param_grid = {
    "nondomin__min_n_assets" : [10],
    "nondomin__threshold" : [-0.5],
    "nondomin__fitness_measures" : [
        [PerfMeasure.MEAN, RiskMeasure.VARIANCE],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.VARIANCE],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.SEMI_VARIANCE],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.MAX_DRAWDOWN],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.VARIANCE],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.SEMI_VARIANCE]
    ],
    "correlate__threshold" : [0.1, 0.2, 0.3],
}
random_search = RandomizedSearchCV(
    estimator=model,
    cv=cv,
    param_distributions=param_grid,
    scoring=make_scorer(PerfMeasure.ANNUALIZED_MEAN),
    n_jobs=12,
    verbose=3,
    n_iter=30,
    refit=False,
)
random_search.fit(X)
print(random_search.best_params_)
print(random_search.best_score_)

#### WalkForward+Optuna

In [16]:
class StopWhenNoImprovement:
    def __init__(self, patience=20, min_delta=1e-4):
        self.patience = patience        # 连续多少次无提升就停止
        self.min_delta = min_delta      # 提升多少才算"有效改善"
        self.best_value = None
        self.no_improve_count = 0

    def __call__(self, study, trial):
        current_value = study.best_value
        
        if self.best_value is None:
            self.best_value = current_value
            return
        
        # 判断是否有有效改善
        if current_value - self.best_value > self.min_delta:
            self.best_value = current_value
            self.no_improve_count = 0
        else:
            self.no_improve_count += 1
        
        if self.no_improve_count >= self.patience:
            print(f"\n连续 {self.patience} 次 trial 无有效改善，提前停止。")
            study.stop()

In [15]:
# MEAN + RiskMeasure
FITNESS_MEASURES = {
    "mean-variance":              [PerfMeasure.MEAN, RiskMeasure.VARIANCE],
    "mean-semivariance":          [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE],
    "mean-std":                   [PerfMeasure.MEAN, RiskMeasure.STANDARD_DEVIATION],
    "mean-semidev":               [PerfMeasure.MEAN, RiskMeasure.SEMI_DEVIATION],
    "mean-mad":                   [PerfMeasure.MEAN, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
    "mean-cvar":                  [PerfMeasure.MEAN, RiskMeasure.CVAR],
    "mean-evar":                  [PerfMeasure.MEAN, RiskMeasure.EVAR],
    "mean-worst-realization":     [PerfMeasure.MEAN, RiskMeasure.WORST_REALIZATION],
    "mean-cdar":                  [PerfMeasure.MEAN, RiskMeasure.CDAR],
    "mean-maxdd":                 [PerfMeasure.MEAN, RiskMeasure.MAX_DRAWDOWN],
    "mean-avgdd":                 [PerfMeasure.MEAN, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-edar":                  [PerfMeasure.MEAN, RiskMeasure.EDAR],
    "mean-first-lpm":             [PerfMeasure.MEAN, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
    "mean-ulcer":                 [PerfMeasure.MEAN, RiskMeasure.ULCER_INDEX],
    "mean-gini":                  [PerfMeasure.MEAN, RiskMeasure.GINI_MEAN_DIFFERENCE],
}

In [20]:
# MEAN + RatioMeasure
FITNESS_MEASURES = {
    "mean-sharpe":              [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO],
    "mean-sortino":             [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO],
    "mean-mad":                 [PerfMeasure.MEAN, RatioMeasure.MEAN_ABSOLUTE_DEVIATION_RATIO],
    "mean-first-lpm":           [PerfMeasure.MEAN, RatioMeasure.FIRST_LOWER_PARTIAL_MOMENT_RATIO],
    "mean-var":                 [PerfMeasure.MEAN, RatioMeasure.VALUE_AT_RISK_RATIO],
    "mean-cvar":                [PerfMeasure.MEAN, RatioMeasure.CVAR_RATIO],
    "mean-entropic":            [PerfMeasure.MEAN, RatioMeasure.ENTROPIC_RISK_MEASURE_RATIO],
    "mean-evar":                [PerfMeasure.MEAN, RatioMeasure.EVAR_RATIO],
    "mean-worst-realization":   [PerfMeasure.MEAN, RatioMeasure.WORST_REALIZATION_RATIO],
    "mean-dar":                 [PerfMeasure.MEAN, RatioMeasure.DRAWDOWN_AT_RISK_RATIO],
    "mean-cdar":                [PerfMeasure.MEAN, RatioMeasure.CDAR_RATIO],
    "mean-calmar":              [PerfMeasure.MEAN, RatioMeasure.CALMAR_RATIO],
    "mean-avgdd":               [PerfMeasure.MEAN, RatioMeasure.AVERAGE_DRAWDOWN_RATIO],
    "mean-edar":                [PerfMeasure.MEAN, RatioMeasure.EDAR_RATIO],
    "mean-ulcer":               [PerfMeasure.MEAN, RatioMeasure.ULCER_INDEX_RATIO],
    "mean-gini":                [PerfMeasure.MEAN, RatioMeasure.GINI_MEAN_DIFFERENCE_RATIO],
}

In [ ]:
# SHARPE + RiskMeasure
FITNESS_MEASURES = {
    "sharpe-variance":              [RatioMeasure.SHARPE_RATIO, RiskMeasure.VARIANCE],
    "sharpe-semivariance":          [RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_VARIANCE],
    "sharpe-std":                   [RatioMeasure.SHARPE_RATIO, RiskMeasure.STANDARD_DEVIATION],
    "sharpe-semidev":               [RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_DEVIATION],
    "sharpe-mad":                   [RatioMeasure.SHARPE_RATIO, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
    "sharpe-cvar":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.CVAR],
    "sharpe-evar":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.EVAR],
    "sharpe-worst-realization":     [RatioMeasure.SHARPE_RATIO, RiskMeasure.WORST_REALIZATION],
    "sharpe-cdar":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.CDAR],
    "sharpe-maxdd":                 [RatioMeasure.SHARPE_RATIO, RiskMeasure.MAX_DRAWDOWN],
    "sharpe-avgdd":                 [RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "sharpe-edar":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.EDAR],
    "sharpe-first-lpm":             [RatioMeasure.SHARPE_RATIO, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
    "sharpe-ulcer":                 [RatioMeasure.SHARPE_RATIO, RiskMeasure.ULCER_INDEX],
    "sharpe-gini":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
}

In [ ]:
# SORTINO + RiskMeasure
FITNESS_MEASURES = {
    "sortino-variance":              [RatioMeasure.SORTINO_RATIO, RiskMeasure.VARIANCE],
    "sortino-semivariance":          [RatioMeasure.SORTINO_RATIO, RiskMeasure.SEMI_VARIANCE],
    "sortino-std":                   [RatioMeasure.SORTINO_RATIO, RiskMeasure.STANDARD_DEVIATION],
    "sortino-semidev":               [RatioMeasure.SORTINO_RATIO, RiskMeasure.SEMI_DEVIATION],
    "sortino-mad":                   [RatioMeasure.SORTINO_RATIO, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
    "sortino-cvar":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.CVAR],
    "sortino-evar":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.EVAR],
    "sortino-worst-realization":     [RatioMeasure.SORTINO_RATIO, RiskMeasure.WORST_REALIZATION],
    "sortino-cdar":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.CDAR],
    "sortino-maxdd":                 [RatioMeasure.SORTINO_RATIO, RiskMeasure.MAX_DRAWDOWN],
    "sortino-avgdd":                 [RatioMeasure.SORTINO_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "sortino-edar":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.EDAR],
    "sortino-first-lpm":             [RatioMeasure.SORTINO_RATIO, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
    "sortino-ulcer":                 [RatioMeasure.SORTINO_RATIO, RiskMeasure.ULCER_INDEX],
    "sortino-gini":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
}

In [ ]:
# CALMAR + RiskMeasure
FITNESS_MEASURES = {
    "calmar-variance":              [RatioMeasure.CALMAR_RATIO, RiskMeasure.VARIANCE],
    "calmar-semivariance":          [RatioMeasure.CALMAR_RATIO, RiskMeasure.SEMI_VARIANCE],
    "calmar-std":                   [RatioMeasure.CALMAR_RATIO, RiskMeasure.STANDARD_DEVIATION],
    "calmar-semidev":               [RatioMeasure.CALMAR_RATIO, RiskMeasure.SEMI_DEVIATION],
    "calmar-mad":                   [RatioMeasure.CALMAR_RATIO, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
    "calmar-cvar":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.CVAR],
    "calmar-evar":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.EVAR],
    "calmar-worst-realization":     [RatioMeasure.CALMAR_RATIO, RiskMeasure.WORST_REALIZATION],
    "calmar-cdar":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.CDAR],
    "calmar-maxdd":                 [RatioMeasure.CALMAR_RATIO, RiskMeasure.MAX_DRAWDOWN],
    "calmar-avgdd":                 [RatioMeasure.CALMAR_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "calmar-edar":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.EDAR],
    "calmar-first-lpm":             [RatioMeasure.CALMAR_RATIO, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
    "calmar-ulcer":                 [RatioMeasure.CALMAR_RATIO, RiskMeasure.ULCER_INDEX],
    "calmar-gini":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
}

In [30]:
# Perf + Risk + Ratio
FITNESS_MEASURES = {
    "mean-sortino-ratio-variance":        [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.VARIANCE],
    "mean-sortino-ratio-avgdd":           [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-sortino-ratio-cdar":            [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.CDAR],
    "mean-sortino-ratio-cvar":            [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.CVAR],
    "mean-sortino-ratio-edar":            [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.EDAR],
    "mean-sortino-ratio-evar":            [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.EVAR],
    "mean-sortino-ratio-ulcer":           [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.ULCER_INDEX],
    "mean-sortino-ratio-gmd":             [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
    "mean-sharpe-ratio-semivariance":     [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_VARIANCE],
    "mean-sharpe-ratio-semivariance":     [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_VARIANCE, ExtraRiskMeasure.KURTOSIS],
    "mean-sharpe-ratio-avgdd":            [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-sharpe-ratio-avgdd-kurt":       [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN, ExtraRiskMeasure.KURTOSIS],
    "mean-sharpe-ratio-cdar":             [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.CDAR],
    "mean-sharpe-ratio-cvar":             [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.CVAR],
    "mean-sharpe-ratio-edar":             [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.EDAR],
    "mean-sharpe-ratio-evar":             [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.EVAR],
    "mean-sharpe-ratio-ulcer":            [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.ULCER_INDEX],
    "mean-sharpe-ratio-gmd":              [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
    "mean-variance-cvar":                 [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.CVAR],
    "mean-variance-kurt":                 [PerfMeasure.MEAN, RiskMeasure.VARIANCE, ExtraRiskMeasure.KURTOSIS],
    "mean-semivariance-cvar":             [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.CVAR],
    "mean-variance-avgdd":                [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-semivariance-avgdd":            [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],

}

In [17]:
# MEAN + RatioMeasure
FITNESS_MEASURES = {
    "mean-variance":            [PerfMeasure.MEAN, RiskMeasure.VARIANCE],
    "mean-variance-sharpe":     [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RatioMeasure.SHARPE_RATIO],
    "mean-variance-cvar":       [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.CVAR],
    "mean-variance-cdar":       [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.CDAR],
    "mean-variance-ulcer":      [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.ULCER_INDEX],
    "mean-variance-avgdd":      [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-variance-gini":       [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.GINI_MEAN_DIFFERENCE],
    "mean-semivariance":        [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE],
    "mean-semivariance-sortino":[PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RatioMeasure.SORTINO_RATIO],
    "mean-semivariance-cvar":   [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.CVAR],
    "mean-semivariance-cdar":   [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.CDAR],
    "mean-semivariance-ulcer":  [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.ULCER_INDEX],
    "mean-semivariance-avgdd":  [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-semivariance-gini":   [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.GINI_MEAN_DIFFERENCE],                
}

In [ ]:
# ---- 1. 复用你原有的组件 ----
cv = WalkForward(
    test_size=252//4,
    train_size=int(252 * 2),
    purged_size=0,
    reduce_test=True,
    expand_train=False,
)

# ---- 2. 定义 objective 函数 ----
def objective(trial):
    #min_n_assets = trial.suggest_categorical("nondomin__min_n_assets", [5, 10, 15, 20, 25])
    #threshold = trial.suggest_categorical("nondomin__threshold", [-0.25, -0.3, -0.4, -0.5])
    #corr_threshold = trial.suggest_categorical("correlate__threshold", [0.1, 0.2, 0.3, 0.4, 0.5])
    # 有序参数：让 TPE 感知数值距离
    min_n_assets = trial.suggest_int("nondomin__min_n_assets", 5, 25, step=5)
    threshold = trial.suggest_float("nondomin__threshold", -0.6, -0.5, step=0.05)
    corr_threshold = trial.suggest_float("correlate__threshold", 0.1, 0.2, step=0.05)
    # 无序分类参数：fitness_measures 是筛选逻辑，不是目标函数
    fm_name = trial.suggest_categorical("nondomin__fitness_measures", list(FITNESS_MEASURES))
    fitness_measures = FITNESS_MEASURES[fm_name]

    model = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated(
            min_n_assets=min_n_assets,
            threshold=threshold,
            fitness_measures=fitness_measures,
        )),
        ("correlate", DropCorrelated(threshold=corr_threshold)),
        ("optimization", EqualWeighted()),
    ])

    pred = cross_val_predict(model, X, cv=cv, n_jobs=8)  # 无 y
    return float(pred.annualized_mean)


# ---- 3. 创建 Study 并运行 ----
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=800,       # 等价于你原来的 n_iter=10
    n_jobs=12,          # ⚠️ trial 级并行，每个 trial 内部串行
    show_progress_bar=True,
    callbacks=[StopWhenNoImprovement(patience=200, min_delta=1e-4)],
)

# ---- 4. 查看结果 ----
print("最优参数：", study.best_params)
print("最优得分：", study.best_value)
# 可选：可视化搜索过程
# optuna.visualization.plot_optimization_history(study).show()

In [32]:
optuna.visualization.plot_optimization_history(study).show()

#### WalkForward + Best Parameter

In [ ]:
最优参数： {'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.25, 'correlate__threshold': 0.4, 'nondomin__fitness_measures': 'mean-gini-ratio-variance'}
最优得分： 0.0898399265461133
MAX Drawdown                              8.57%
Average Drawdown                          2.34%
Annualized Sharpe Ratio                    1.27
Annualized Sortino Ratio                   1.75

126, 252*2
最优参数： {'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.1, 'nondomin__fitness_measures': 'mean-gini-ratio-semivariance'}
最优得分： 0.10591870482755297
MAX Drawdown                              9.59%
Average Drawdown                          1.83%
Annualized Sharpe Ratio                    1.21
Annualized Sortino Ratio                   1.73

最优参数： {'nondomin__min_n_assets': 15, 'nondomin__threshold': -0.25, 'correlate__threshold': 0.1, 'nondomin__fitness_measures': 'mean-sharpe-ratio-semivariance'}
最优得分： 0.10622994041257404
MAX Drawdown                             17.06%
Average Drawdown                          3.65%
Annualized Sharpe Ratio                    1.19
Annualized Sortino Ratio                   1.69

最优参数： {'nondomin__min_n_assets': 20, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.1, 'nondomin__fitness_measures': 'mean-sharpe-ratio-avgdd'}
最优得分： 0.09875816355755006

#test_size=252//2,train_size=int(252 * 3),
最优参数： {'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.5, 'nondomin__fitness_measures': 'mean-sharpe-ratio-cvar'}
最优得分： 0.09073300958104169
Skew                                    -47.04%
Kurtosis                                798.35%
MAX Drawdown                             11.92%
Average Drawdown                          4.23%
#test_size=252//2,train_size=int(252 * 2),
最优参数： {'nondomin__min_n_assets': 15, 'nondomin__threshold': -0.3, 'correlate__threshold': 0.1, 'nondomin__fitness_measures': 'mean-sortino-ratio-edar'}
最优得分： 0.10564773551492695

{'correlate__threshold': 0.3, 'nondomin__fitness_measures': [Mean, Variance], 'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.3}
0.07908080360495395

{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Semi-Variance], 'nondomin__min_n_assets': 10, 'nondomin__threshold': -0.5}
0.07962022084135686

{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Mean Absolute Deviation], 'nondomin__min_n_assets': 10, 'nondomin__threshold': -0.5}
0.07390850228424162

{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Semi-Variance], 'nondomin__min_n_assets': 15, 'nondomin__threshold': -0.5}
0.15097108574668483

252//2 252*2
{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Average Drawdown], 'nondomin__min_n_assets': 10, 'nondomin__threshold': -0.5}
0.09893299218882344

{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Value at Risk Ratio], 'nondomin__min_n_assets': 10, 'nondomin__threshold': -0.5}
0.04736766522606962

{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Variance, Average Drawdown], 'nondomin__min_n_assets': 10, 'nondomin__threshold': -0.5}
0.21115761008572798

{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Variance, Average Drawdown], 'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.5}
0.21115761008572798

252//4 252*2
{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Variance, Average Drawdown], 'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.4}
0.06535492444170049

{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Sharpe Ratio, Ulcer Index], 'nondomin__min_n_assets': 10, 'nondomin__threshold': -0.5}
0.2437614863939793

252//4 252*1
{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Variance, Average Drawdown], 'nondomin__min_n_assets': 20, 'nondomin__threshold': -0.4}
-0.06120336046338341

{'correlate__threshold': 0.1, 'nondomin__fitness_measures': [Mean, Sharpe Ratio, EVaR], 'nondomin__min_n_assets': 10, 'nondomin__threshold': -0.5}
0.02792606969781052

In [70]:
selection_pipe = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated(min_n_assets=5, threshold=-0.5,
                        fitness_measures=[PerfMeasure.MEAN,
                        RiskMeasure.VARIANCE,
                        RiskMeasure.AVERAGE_DRAWDOWN
                        ])),
        ("correlate", DropCorrelated(threshold=0.1, absolute=False)),
    ])

In [71]:
train_portfolios = []
test_portfolios = []
train_portfolios_net = []
test_portfolios_net = []

cv = WalkForward(test_size=252//2, train_size=int(252*2), purged_size=1, reduce_test=True, expand_train=False)
for i, (train_index, test_index) in enumerate(cv.split(X_net)):
    # 划分训练测试集（X 与 X_net 行索引一致）
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    X_net_train = X_net.iloc[train_index]
    X_net_test = X_net.iloc[test_index]

    # 筛选在超额口径 X_net 上执行，列集由 X_net 决定
    X_train = selection_pipe.fit_transform(X_train)
    cols = X_train.columns
    if len(cols) == 0:
        continue

    # 训练在超额口径 X_net 上执行（等权权重与数值无关，仅记录列集）
    m = EqualWeighted(portfolio_params=dict(name="Fold %d" % i)).fit(X_train)

    # X 口径：真实收益
    train_portfolios.append(m.predict(X_train[cols]))
    test_portfolios.append(m.predict(X_test[cols]))
    # X_net 口径：超额收益
    train_portfolios_net.append(m.predict(X_train))
    test_portfolios_net.append(m.predict(X_net_test[cols]))

population_train = Population(train_portfolios)
population_test = Population(MultiPeriodPortfolio(test_portfolios))
population_train_net = Population(train_portfolios_net)
population_test_net = Population(MultiPeriodPortfolio(test_portfolios_net))

population_train.set_portfolio_params(tag="Train")
population_test.set_portfolio_params(tag="Test")
population_train_net.set_portfolio_params(tag="Train (Net)")
population_test_net.set_portfolio_params(tag="Test (Net)")
population = population_train + population_test
population_net = population_train_net + population_test_net

In [72]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [73]:
population_test.plot_cumulative_returns()

In [74]:
MultiPeriodPortfolio(test_portfolios).plot_cumulative_returns()

In [75]:
MultiPeriodPortfolio(test_portfolios).summary()

Mean                                     0.033%
Annualized Mean                           8.42%
Variance                                0.0021%
Annualized Variance                       0.53%
Semi-Variance                          0.00098%
Annualized Semi-Variance                  0.25%
Standard Deviation                        0.46%
Annualized Standard Deviation             7.26%
Semi-Deviation                            0.31%
Annualized Semi-Deviation                 4.98%
Mean Absolute Deviation                   0.29%
CVaR at 95%                               1.04%
EVaR at 95%                               1.74%
Worst Realization                         3.09%
CDaR at 95%                               6.13%
MAX Drawdown                              8.30%
Average Drawdown                          1.61%
EDaR at 95%                               6.57%
First Lower Partial Moment                0.14%
Ulcer Index                               0.022
Gini Mean Difference                    